# Interactive Diffusion & Latent Space Simulator (Google Colab T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChiVan222/AI_Seminar_Diffusion/blob/main/experiments/diffusion_simulator_colab.ipynb)

Launch the interactive **Streamlit Diffusion Simulator** powered by pretrained **Stable Diffusion v1.5 FP16** (`stable-diffusion-v1-5/stable-diffusion-v1-5`) and **Pixel DDPM CelebA-HQ 256** (`google/ddpm-celebahq-256`) on a free Google Colab **Tesla T4 GPU**:

1. **Tab 1 — Unconditioned (∅) vs. Conditioned LDM (CFG):** Run both trajectories from the exact same initial noise $\mathbf{z}_T$, scrub step-by-step, inspect CFG guidance maps $\|\epsilon_\text{cond} - \epsilon_\varnothing\|_2$, and compare 2D PCA paths.
2. **Tab 2 — Step-by-Step Latent Space Inspector:** Inspect all 4 latent channels `z[0..3]`, linear RGB latent projections, VAE decoded $\mathcal{D}(\mathbf{z}_t)$, predicted clean $\mathcal{D}(\hat{\mathbf{z}}_0)$, and distributions vs. $\mathcal{N}(0, 1)$.
3. **Tab 3 — Forward vs. Backward DDPM (Latent Space):** Compare Closed-Form jump vs. Step-by-Step Markov noising ($0 \to t_{\max}$) and Unconditioned vs. Conditioned reverse denoising ($t_{\max} \to 0$).
4. **Tab 4 — Forward vs. Backward DDPM (Pixel Space) & Benchmark:** Run `google/ddpm-celebahq-256` directly on $256\times 256\times 3$ pixels and compare latency/VRAM/dimensionality against Latent Diffusion.

> **Note:** In Colab, select **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# 1. Verify Colab Tesla T4 GPU & CUDA availability
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA Ready | Device: {name} | VRAM: {vram:.1f} GB")
else:
    print("WARNING: No GPU detected! Switch Runtime -> Change runtime type -> T4 GPU.")

!nvidia-smi

In [ ]:
# 2. Clone or update repository
import os
import subprocess

REPO_URL = "https://github.com/ChiVan222/AI_Seminar_Diffusion.git"
REPO_DIR = "/content/AI_Seminar_Diffusion"

if os.path.exists("experiments/diffusion_simulator/app.py"):
    REPO_DIR = os.getcwd()
elif not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
assert os.path.exists("experiments/diffusion_simulator/app.py"), (
    "experiments/diffusion_simulator/app.py not found. Make sure the latest commit is pushed to GitHub!"
)
print("Working directory:", os.getcwd())

In [ ]:
# 3. Install Python dependencies & cloudflared tunnel binary (passwordless HTTPS)
!pip install -q --upgrade diffusers transformers accelerate safetensors streamlit pillow imageio scikit-learn matplotlib scipy

import os
import subprocess

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        [
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", "/usr/local/bin/cloudflared",
        ],
        check=True,
    )
    os.chmod("/usr/local/bin/cloudflared", 0o755)
print("Dependencies and cloudflared ready!")

In [ ]:
# 4. Optional: Pre-download pretrained weights to Colab disk (~2.4 GB total) for instant UI startup
import sys
sys.path.insert(0, os.path.join(os.getcwd(), "experiments"))

from diffusion_simulator.models import load_sd15_pipeline, load_pixel_ddpm_pipeline, free_gpu_memory

print("Pre-caching Stable Diffusion v1.5 (FP16)...")
sd_pipe = load_sd15_pipeline()
del sd_pipe
free_gpu_memory()

print("Pre-caching Pixel DDPM CelebA-HQ 256...")
ddpm_pipe = load_pixel_ddpm_pipeline()
del ddpm_pipe
free_gpu_memory()
print("All pretrained models cached cleanly in Hugging Face hub cache!")

In [ ]:
# 5. Launch Streamlit Simulator + Public Cloudflare HTTPS Tunnel
import re
import subprocess
import time
import urllib.request
from IPython.display import HTML, display

# Clean up any previous instances on port 8501
subprocess.run(["pkill", "-f", "streamlit"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(1)

st_log = open("/tmp/streamlit_sim.log", "w")
st_proc = subprocess.Popen(
    [
        "streamlit", "run", "experiments/diffusion_simulator/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    stdout=st_log,
    stderr=subprocess.STDOUT,
)

# Wait for local Streamlit health check
ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    st_proc.terminate()
    raise RuntimeError("Streamlit failed to start. Inspect /tmp/streamlit_sim.log")

cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

tunnel_url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = cf_proc.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    display(HTML(
        f"<div style='padding:14px;border-radius:8px;background:#ECF2FA;border-left:5px solid #19468C;'>"
        f"<h3 style='margin:0;color:#19468C;'>Diffusion Simulator Live on Colab T4</h3>"
        f"<p style='margin:8px 0 0 0;'>Open UI: <a href='{tunnel_url}' target='_blank' style='font-weight:bold;font-size:16px;'>{tunnel_url}</a></p>"
        f"<p style='margin:6px 0 0 0;color:#475569;font-size:12px;'>Keep this cell running while using the app. Stop this cell to shut down the server.</p>"
        f"</div>"
    ))
else:
    print("Cloudflare tunnel URL not detected automatically. Fallback: run google.colab.output.serve_kernel_port_as_window(8501)")

try:
    st_proc.wait()
except KeyboardInterrupt:
    print("Shutting down Streamlit and tunnel...")
finally:
    cf_proc.terminate()
    st_proc.terminate()
    st_log.close()